---
## Data Loading & Inspection

In [ ]:
import pandas as pd
import numpy as np

# Muat dataset
df = pd.read_csv('dataset_penjualan_kantin.csv', dtype=str)

print('✅ Dataset berhasil dimuat!')
print(f'Jumlah baris x kolom: {df.shape}')

In [ ]:
# Tampilkan 5 baris pertama
print('--- HEAD (5 baris pertama) ---')
df.head()

In [ ]:
# Informasi struktur DataFrame
print('--- INFO ---')
df.info()

In [ ]:
# Statistik deskriptif (semua kolom masih string, akan dikonversi saat cleaning)
print('--- DESCRIBE ---')
df.describe(include='all')

In [ ]:
# Identifikasi masalah kualitas data
print('--- MISSING VALUES per kolom ---')
print(df.isnull().sum())

print(f'\n--- DUPLIKAT ---')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

### Ringkasan Temuan Inspeksi Awal
| Masalah | Detail |
|---|---|
| **Missing Value** | `jumlah_terjual` (4 baris), `harga_satuan` (3 baris), `nama_kasir` (3 baris) |
| **Duplikat** | 4 pasang baris duplikat ditemukan (8 baris total) |
| **Tipe Data** | Semua kolom bertipe `object/str`; `jumlah_terjual` & `harga_satuan` perlu dikonversi ke angka |
| **Format Tidak Konsisten** | Kolom `tanggal` memiliki berbagai format (ISO, dd/mm/yyyy, "14 Agustus 2026") |
| **Nilai Tidak Wajar** | `harga_satuan` ada yang bertipe "Rp2.000" (ada prefix Rp & titik ribuan); `jumlah_terjual` ada "12 pcs" (ada suffix teks); satu baris `jumlah_terjual` = 500 (kemungkinan outlier) |
| **Inkonsistensi Kategori** | Kolom `kategori` campur huruf besar-kecil (makanan/MAKANAN/Makanan) |

---
## Data Cleaning

### 2a. Tangani Duplikat
**Alasan:** Baris yang sepenuhnya identik (semua kolom sama) tidak menambah informasi baru, sehingga cukup dibuang dengan `drop_duplicates()`. Mempertahankan duplikat akan membuat agregasi (total/rata-rata) menjadi tidak akurat.

In [ ]:
print(f'Baris sebelum hapus duplikat: {len(df)}')
df = df.drop_duplicates()
print(f'Baris setelah hapus duplikat : {len(df)}')
print(f'Duplikat tersisa: {df.duplicated().sum()}')

### 2b. Bersihkan & Konversi Kolom `harga_satuan`
**Alasan:** Nilai seperti `"Rp2.000"` tidak bisa langsung dioperasikan secara matematis. Fungsi pembersih menghapus semua karakter non-digit, lalu nilai yang masih kosong/tidak valid **di-drop (`dropna`)** karena harga adalah kolom kunci, kita tidak bisa mengisi harga yang tidak diketahui dengan angka sembarang tanpa menyesatkan analisis.

In [ ]:
def bersihkan_angka(nilai):
    """Menghapus karakter non-digit (Rp, titik, koma, spasi) lalu konversi ke int."""
    if pd.isna(nilai):
        return np.nan
    hasil = ''.join(c for c in str(nilai) if c.isdigit())
    return int(hasil) if hasil else np.nan

df['harga_satuan'] = df['harga_satuan'].apply(bersihkan_angka)

# Drop baris yang harga_satuan-nya tidak bisa diselamatkan
sebelum = len(df)
df = df.dropna(subset=['harga_satuan'])
print(f'Baris di-drop karena harga_satuan kosong: {sebelum - len(df)}')

df['harga_satuan'] = df['harga_satuan'].astype(int)
print('Contoh nilai harga_satuan setelah dibersihkan:')
print(df['harga_satuan'].value_counts().sort_index())

### 2c. Bersihkan & Konversi Kolom `jumlah_terjual`
**Alasan:** Nilai seperti `"12 pcs"` atau `"7 pcs"` mengandung suffix teks yang harus dibuang. Untuk nilai yang benar-benar kosong (NaN), digunakan **`fillna(0)`** karena missing di sini bisa diinterpretasikan sebagai transaksi yang tercatat tapi jumlahnya belum diisi, mengisi dengan 0 lebih aman daripada menghapus seluruh transaksi.

In [ ]:
df['jumlah_terjual'] = df['jumlah_terjual'].apply(bersihkan_angka)
df['jumlah_terjual'] = df['jumlah_terjual'].fillna(0).astype(int)

print('Nilai unik jumlah_terjual setelah dibersihkan:')
print(sorted(df['jumlah_terjual'].unique()))

### 2d. Tangani Missing Value `nama_kasir`
**Alasan:** `nama_kasir` yang kosong tidak mempengaruhi analisis penjualan utama. Daripada menghapus baris transaksi yang valid, nilai diisi dengan `"Tidak Diketahui"` sebagai penanda data tidak lengkap.

In [ ]:
print(f'Missing nama_kasir sebelum fillna: {df["nama_kasir"].isna().sum()}')
df['nama_kasir'] = df['nama_kasir'].fillna('Tidak Diketahui')
print(f'Missing nama_kasir setelah fillna : {df["nama_kasir"].isna().sum()}')

### 2e. Standardisasi Kolom `kategori` & Konversi `tanggal`
**Alasan:** Inkonsistensi kapitalisasi (makanan/MAKANAN/Makanan) akan menyebabkan groupby menghasilkan kategori ganda yang salah. `.str.capitalize()` menyeragamkan ke format "Title case per kata". Kolom `tanggal` dikonversi ke datetime agar bisa diurutkan dan diekstrak hari/bulannya.

In [ ]:
# Standardisasi kategori
df['kategori'] = df['kategori'].str.strip().str.capitalize()
print('Nilai unik kategori:', sorted(df['kategori'].unique()))

import re

# Konversi tanggal
bulan_id = {
    'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
    'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
    'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

def konversi_tanggal(tgl):
    """Normalisasi berbagai format tanggal ke format dd/mm/yyyy."""
    tgl = str(tgl).strip()
    # Ganti nama bulan ke angka
    for nama, angka in bulan_id.items():
        tgl = tgl.replace(nama, angka)
    # Normalisasi '14 08 2026' → '14/08/2026' (hasil replace nama bulan + spasi)
    m = re.match(r'^(\d{1,2})\s+(\d{2})\s+(\d{4})$', tgl)
    if m:
        d, mo, y = m.groups()
        tgl = f'{int(d):02d}/{mo}/{y}'
    return tgl

df['tanggal'] = df['tanggal'].apply(konversi_tanggal)
df['tanggal'] = pd.to_datetime(df['tanggal'], dayfirst=True, errors='coerce')

print(f'Missing tanggal setelah konversi: {df["tanggal"].isna().sum()}')
print(f'Rentang tanggal: {df["tanggal"].min().date()} s.d. {df["tanggal"].max().date()}')

### 2f. Verifikasi Tipe Data Akhir

In [ ]:
print('--- TIPE DATA SETELAH CLEANING ---')
print(df.dtypes)

print('\n--- MISSING VALUES AKHIR ---')
print(df.isnull().sum())

print(f'\nShape dataset bersih: {df.shape}')

In [ ]:
# Cek potensi outlier pada jumlah_terjual
print('--- STATISTIK jumlah_terjual ---')
print(df['jumlah_terjual'].describe())
print('\nBaris dengan jumlah_terjual > 50 (potensi tidak wajar):')
print(df[df['jumlah_terjual'] > 50][['id_transaksi','nama_produk','jumlah_terjual','harga_satuan']])

 **Catatan Outlier:** Baris `TRX0050` memiliki `jumlah_terjual = 500` untuk Roti Bakar, jauh melampaui nilai lain (maks normal ~15). Nilai ini **dipertahankan** di dataset bersih sebagai temuan analisis yang perlu dikonfirmasi ke sumber data, namun akan dicatat dalam ringkasan.

---
## Data Manipulation

### 3a. Kolom Turunan: `total_pendapatan` & `hari`

In [ ]:
# Kolom turunan 1: total pendapatan per transaksi
df['total_pendapatan'] = df['jumlah_terjual'] * df['harga_satuan']

# Kolom turunan 2: nama hari transaksi
hari_map = {0:'Senin',1:'Selasa',2:'Rabu',3:'Kamis',4:'Jumat',5:'Sabtu',6:'Minggu'}
df['hari'] = df['tanggal'].dt.dayofweek.map(hari_map)

print('Kolom baru berhasil dibuat:')
df[['id_transaksi','nama_produk','jumlah_terjual','harga_satuan','total_pendapatan','hari']].head(8)

### 3b. Filtering: Transaksi dengan Metode Pembayaran QRIS

In [ ]:
df_qris = df[df['metode_pembayaran'] == 'QRIS'].copy()
print(f'Jumlah transaksi QRIS: {len(df_qris)} dari {len(df)} total transaksi')
df_qris[['id_transaksi','tanggal','nama_produk','jumlah_terjual','total_pendapatan']].head()

### 3c. Sorting: Transaksi Diurutkan dari Total Pendapatan Terbesar

In [ ]:
df_sorted = df.sort_values(by='total_pendapatan', ascending=False).reset_index(drop=True)
print('10 Transaksi dengan total pendapatan terbesar:')
df_sorted[['id_transaksi','tanggal','nama_produk','jumlah_terjual','harga_satuan','total_pendapatan']].head(10)

### 3d. GroupBy & Agregasi: Penjualan per Produk

In [ ]:
# Agregasi 1: total unit terjual & total pendapatan per produk
agg_produk = df.groupby('nama_produk').agg(
    total_unit=('jumlah_terjual', 'sum'),
    total_pendapatan=('total_pendapatan', 'sum'),
    harga_satuan=('harga_satuan', 'first')
).reset_index().sort_values('total_unit', ascending=False)

print('--- Peringkat Produk berdasarkan Total Unit Terjual ---')
print(agg_produk.to_string(index=False))

In [ ]:
# Agregasi 2: total pendapatan per kategori
agg_kategori = df.groupby('kategori').agg(
    total_unit=('jumlah_terjual', 'sum'),
    total_pendapatan=('total_pendapatan', 'sum'),
    jumlah_transaksi=('id_transaksi', 'count')
).reset_index().sort_values('total_pendapatan', ascending=False)

print('--- Pendapatan per Kategori ---')
print(agg_kategori.to_string(index=False))

In [ ]:
# Agregasi 3: performa per kasir
agg_kasir = df.groupby('nama_kasir').agg(
    total_pendapatan=('total_pendapatan', 'sum'),
    jumlah_transaksi=('id_transaksi', 'count')
).reset_index().sort_values('total_pendapatan', ascending=False)

print('--- Performa per Kasir ---')
print(agg_kasir.to_string(index=False))

---
## Simpan Dataset Bersih

In [ ]:
df.to_csv('dataset_bersih.csv', index=False)
print(f'Dataset bersih disimpan ke: dataset_bersih.csv')
print(f'Jumlah baris: {len(df)}, Jumlah kolom: {len(df.columns)}')
print(f'Kolom: {list(df.columns)}')


### Temuan 1: Produk Terlaris: Roti Bakar (dengan catatan)
Jika dihitung dari total unit terjual, **Roti Bakar** tampak sebagai produk dengan angka tertinggi. Namun, angka tersebut sangat dipengaruhi oleh satu transaksi mencurigakan (`TRX0050`) yang mencatat penjualan **500 pcs** sekaligus jauh di atas transaksi lain yang umumnya berkisar 1–15 unit. Jika baris outlier ini dikesampingkan, **Jus Alpukat** dan **Nasi Goreng** muncul sebagai produk yang paling konsisten laris berdasarkan frekuensi transaksi dan akumulasi unit yang masuk akal. Data ini perlu dikonfirmasi ulang ke sumber untuk memastikan apakah 500 adalah kesalahan input atau memang pesanan dalam jumlah besar.

### Temuan 2: Produk Termurah: Kerupuk & Gorengan (Rp2.000)
Produk dengan harga satuan terendah adalah **Kerupuk** dan **Gorengan**, masing-masing seharga **Rp2.000**. Meskipun murah, kedua produk ini tetap memiliki frekuensi transaksi yang cukup sering. Produk paling mahal adalah **Nasi Goreng** dan **Bakso** (Rp11.000–Rp12.000), yang juga merupakan produk dengan kontribusi total pendapatan per transaksi paling tinggi di kategori Makanan.

### Temuan 3: Kualitas Data Memerlukan Perhatian Serius
Dataset awal mengandung sejumlah masalah kualitas yang cukup signifikan: **4 baris duplikat**, **10 missing value** tersebar di tiga kolom, serta inkonsistensi format pada kolom `tanggal` (tiga format berbeda) dan `kategori` (enam variasi kapitalisasi). Setelah proses cleaning, dataset berkurang dari 69 menjadi **65 baris bersih** (4 duplikat dihapus, 2 baris tanpa harga satuan di-drop). Temuan ini menunjukkan bahwa sistem pencatatan transaksi kantin perlu distandarisasi, idealnya menggunakan formulir digital atau kasir otomatis agar data lebih konsisten sejak awal.